In [26]:
import requests
import pandas as pd
from pathlib import Path

BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"

def fetch_all_verslagen():
    rows = []
    skip = 0

    while True:
        params = {
            "$filter": "Verwijderd eq false",
            "$select": "Id,Vergadering_Id,Soort,Status",
            "$top": 250,
            "$skip": skip
        }

        r = requests.get(f"{BASE}/Verslag", params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("value", [])

        if not batch:
            break

        rows.extend(batch)
        skip += 250

    return pd.DataFrame(rows)

verslag_df = fetch_all_verslagen()
print(verslag_df.head())
print(len(verslag_df))

                                     Id             Soort          Status  \
0  94fbdc2b-fd65-4610-a340-000018d72027  Tussenpublicatie  Ongecorrigeerd   
1  d25cda29-31db-4f99-8ffc-0006a9dd9be5  Tussenpublicatie  Ongecorrigeerd   
2  571d1053-faaa-41f0-b61b-000fb7384c2c    Voorpublicatie           Casco   
3  3c686811-8a3d-4213-88eb-00155c753f29    Voorpublicatie           Casco   
4  57f69332-8f44-4e5a-a6c2-0018384cd49f    Voorpublicatie           Casco   

                         Vergadering_Id  
0  3cb69a3e-7f81-404f-bdcd-6d7e64638f54  
1  9d5e49d8-11d4-4034-a937-9cb61e22357f  
2  625b554a-fd34-4988-b828-3dfc0a092af4  
3  46059f48-29dc-4e72-9b3b-5df4d7487483  
4  6ded85e6-6135-4920-afcd-fe12b08cf14d  
22603


In [27]:
import xml.etree.ElementTree as ET
import pandas as pd
import re

NS = {"tk": "http://www.tweedekamer.nl/ggm/vergaderverslag/v1.0"}

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_text_from_node(node):
    # grabs all descendant text, including nested nadruk/dossiernummer/etc.
    parts = []
    for t in node.itertext():
        t = clean_text(t)
        if t:
            parts.append(t)
    return clean_text(" ".join(parts))

def parse_speeches_from_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []

    vergadering = root.find("tk:vergadering", NS)
    vergadering_titel = vergadering.findtext("tk:titel", default="", namespaces=NS) if vergadering is not None else ""
    vergadering_datum = vergadering.findtext("tk:datum", default="", namespaces=NS) if vergadering is not None else ""
    vergadering_soort = vergadering.attrib.get("soort", "") if vergadering is not None else ""
    vergadering_objectid = vergadering.attrib.get("objectid", "") if vergadering is not None else ""

    # each main activity
    for activiteit in root.findall(".//tk:activiteit", NS):
        activiteit_soort = activiteit.attrib.get("soort", "")
        activiteit_id = activiteit.attrib.get("objectid", "")
        activiteit_titel = activiteit.findtext("tk:titel", default="", namespaces=NS)
        activiteit_onderwerp = activiteit.findtext("tk:onderwerp", default="", namespaces=NS)

        # each speech turn
        for activiteitdeel in activiteit.findall(".//tk:activiteitdeel[@soort='Spreekbeurt']", NS):
            activiteitdeel_id = activiteitdeel.attrib.get("objectid", "")
            activiteitdeel_titel = activiteitdeel.findtext("tk:titel", default="", namespaces=NS)
            deel_begin = activiteitdeel.findtext("tk:markeertijdbegin", default="", namespaces=NS)
            deel_eind = activiteitdeel.findtext("tk:markeertijdeind", default="", namespaces=NS)

            # there can be multiple woordvoerder nodes in one spreekbeurt
            for woordvoerder in activiteitdeel.findall(".//tk:woordvoerder", NS):
                spreker = woordvoerder.find("tk:spreker", NS)

                spreker_id = spreker.attrib.get("objectid", "") if spreker is not None else ""
                spreker_soort = spreker.attrib.get("soort", "") if spreker is not None else ""
                spreker_naam = spreker.findtext("tk:weergavenaam", default="", namespaces=NS) if spreker is not None else ""
                spreker_verslagnaam = spreker.findtext("tk:verslagnaam", default="", namespaces=NS) if spreker is not None else ""
                spreker_voornaam = spreker.findtext("tk:voornaam", default="", namespaces=NS) if spreker is not None else ""
                spreker_achternaam = spreker.findtext("tk:achternaam", default="", namespaces=NS) if spreker is not None else ""
                spreker_fractie = spreker.findtext("tk:fractie", default="", namespaces=NS) if spreker is not None else ""
                spreker_functie = spreker.findtext("tk:functie", default="", namespaces=NS) if spreker is not None else ""

                begin = woordvoerder.findtext("tk:markeertijdbegin", default="", namespaces=NS)
                eind = woordvoerder.findtext("tk:markeertijdeind", default="", namespaces=NS)
                isvoorzitter = woordvoerder.findtext("tk:isvoorzitter", default="", namespaces=NS)
                isdraad = woordvoerder.findtext("tk:isdraad", default="", namespaces=NS)

                tekst_node = woordvoerder.find("tk:tekst", NS)
                speech_text = extract_text_from_node(tekst_node) if tekst_node is not None else ""

                if speech_text:
                    rows.append({
                        "vergadering_id": vergadering_objectid,
                        "vergadering_soort": vergadering_soort,
                        "vergadering_titel": vergadering_titel,
                        "vergadering_datum": vergadering_datum,
                        "activiteit_id": activiteit_id,
                        "activiteit_soort": activiteit_soort,
                        "activiteit_titel": activiteit_titel,
                        "activiteit_onderwerp": activiteit_onderwerp,
                        "spreekbeurt_id": activiteitdeel_id,
                        "spreekbeurt_titel": activiteitdeel_titel,
                        "spreekbeurt_begin": deel_begin,
                        "spreekbeurt_eind": deel_eind,
                        "speaker_id": spreker_id,
                        "speaker_type": spreker_soort,
                        "speaker_name": spreker_naam,
                        "speaker_report_name": spreker_verslagnaam,
                        "speaker_first_name": spreker_voornaam,
                        "speaker_last_name": spreker_achternaam,
                        "speaker_party": spreker_fractie,
                        "speaker_role": spreker_functie,
                        "speech_begin": begin,
                        "speech_end": eind,
                        "is_voorzitter": isvoorzitter,
                        "is_draad": isdraad,
                        "speech_text": speech_text,
                        "source_xml": str(xml_path),
                    })

    return pd.DataFrame(rows)

In [28]:
def parse_all_xmls(xml_dir="verslagen_xml"):
    frames = []

    for xml_file in Path(xml_dir).glob("*.xml"):
        try:
            df = parse_speeches_from_xml(xml_file)
            if not df.empty:
                frames.append(df)
        except Exception as e:
            print(f"Error in {xml_file}: {e}")

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

speeches_df = parse_all_xmls("verslagen_xml")

In [29]:
print(len(speeches_df))
speeches_df.head()

54


,vergadering_id,vergadering_soort,vergadering_titel,vergadering_datum,activiteit_id,activiteit_soort,activiteit_titel,activiteit_onderwerp,spreekbeurt_id,spreekbeurt_titel,...,speaker_first_name,speaker_last_name,speaker_party,speaker_role,speech_begin,speech_end,is_voorzitter,is_draad,speech_text,source_xml
0,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,3f1096ae-6c45-4b35-89c9-c0e92feb4671,Opening,Opening,Opening,906b4f6d-4b01-42c6-84f3-3b434c46b2bd,Spreekbeurt - De voorzitter,...,Khadija,Arib,PvdA,lid Tweede Kamer,2020-10-13T14:01:13,2020-10-13T14:01:27,true,true,De voorzitter : Ik open de vergadering van de ...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
1,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,75a12a82-abe6-4fc2-a471-92e1ea0b65de,Spreekbeurt - De voorzitter,...,Khadija,Arib,PvdA,lid Tweede Kamer,2020-10-13T14:01:28,2020-10-13T14:01:48,true,false,De voorzitter : We beginnen zoals gebruikelijk...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
2,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,4703af9a-47ab-4e47-9bc2-a601d0da7352,Spreekbeurt - Lodders,...,Helma,Lodders,VVD,lid Tweede Kamer,2020-10-13T14:01:48,2020-10-13T14:03:41,false,false,Mevrouw Lodders (VVD): Voorzitter. In Nederlan...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
3,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,eb1cf359-d659-4877-b23d-8c23152a1a44,Spreekbeurt - Staatssecretaris Vijlbrief,...,Hans,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:03:41,2020-10-13T14:07:49,false,false,"Staatssecretaris Vijlbrief : Voorzitter, dank ...",verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
4,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,eb1cf359-d659-4877-b23d-8c23152a1a44,Spreekbeurt - Staatssecretaris Vijlbrief,...,Hans,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:07:49,2020-10-13T14:08:58,false,false,Staatssecretaris Vijlbrief : Voorzitter. Laat ...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...


In [30]:
from pathlib import Path

xml_files = list(Path("verslagen_xml").glob("*.xml"))
print("XML files found:", len(xml_files))
print(xml_files[:3])

df_test = parse_speeches_from_xml(xml_files[0])
print("Rows in first XML:", len(df_test))
df_test.head()

XML files found: 3
[WindowsPath('verslagen_xml/571d1053-faaa-41f0-b61b-000fb7384c2c.xml'), WindowsPath('verslagen_xml/94fbdc2b-fd65-4610-a340-000018d72027.xml'), WindowsPath('verslagen_xml/d25cda29-31db-4f99-8ffc-0006a9dd9be5.xml')]
Rows in first XML: 0


""


In [31]:
print(len(verslag_df))
verslag_df.head()

22603


,Id,Soort,Status,Vergadering_Id
0,94fbdc2b-fd65-4610-a340-000018d72027,Tussenpublicatie,Ongecorrigeerd,3cb69a3e-7f81-404f-bdcd-6d7e64638f54
1,d25cda29-31db-4f99-8ffc-0006a9dd9be5,Tussenpublicatie,Ongecorrigeerd,9d5e49d8-11d4-4034-a937-9cb61e22357f
2,571d1053-faaa-41f0-b61b-000fb7384c2c,Voorpublicatie,Casco,625b554a-fd34-4988-b828-3dfc0a092af4
3,3c686811-8a3d-4213-88eb-00155c753f29,Voorpublicatie,Casco,46059f48-29dc-4e72-9b3b-5df4d7487483
4,57f69332-8f44-4e5a-a6c2-0018384cd49f,Voorpublicatie,Casco,6ded85e6-6135-4920-afcd-fe12b08cf14d


In [32]:
from pathlib import Path
import requests

BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"

def download_xmls(verslag_df, out_dir="verslagen_xml", limit=3):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    print("Saving to:", out.resolve())

    for i, row in verslag_df.iterrows():
        if limit is not None and i >= limit:
            break

        verslag_id = row["Id"]
        url = f"{BASE}/Verslag({verslag_id})/resource"
        print("Downloading:", verslag_id)

        r = requests.get(url, timeout=120)
        r.raise_for_status()

        filepath = out / f"{verslag_id}.xml"
        filepath.write_bytes(r.content)
        print("Saved:", filepath)

download_xmls(verslag_df, out_dir="verslagen_xml", limit=3)

Saving to: C:\Users\wansc\Downloads\verslagen_xml
Downloading: 94fbdc2b-fd65-4610-a340-000018d72027
Saved: verslagen_xml\94fbdc2b-fd65-4610-a340-000018d72027.xml
Downloading: d25cda29-31db-4f99-8ffc-0006a9dd9be5
Saved: verslagen_xml\d25cda29-31db-4f99-8ffc-0006a9dd9be5.xml
Downloading: 571d1053-faaa-41f0-b61b-000fb7384c2c
Saved: verslagen_xml\571d1053-faaa-41f0-b61b-000fb7384c2c.xml


In [33]:
list(Path("verslagen_xml").glob("*.xml"))

[WindowsPath('verslagen_xml/571d1053-faaa-41f0-b61b-000fb7384c2c.xml'),
 WindowsPath('verslagen_xml/94fbdc2b-fd65-4610-a340-000018d72027.xml'),
 WindowsPath('verslagen_xml/d25cda29-31db-4f99-8ffc-0006a9dd9be5.xml')]

In [34]:
speeches_df = parse_all_xmls("verslagen_xml")
print(len(speeches_df))
speeches_df.head()

54


,vergadering_id,vergadering_soort,vergadering_titel,vergadering_datum,activiteit_id,activiteit_soort,activiteit_titel,activiteit_onderwerp,spreekbeurt_id,spreekbeurt_titel,...,speaker_first_name,speaker_last_name,speaker_party,speaker_role,speech_begin,speech_end,is_voorzitter,is_draad,speech_text,source_xml
0,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,3f1096ae-6c45-4b35-89c9-c0e92feb4671,Opening,Opening,Opening,906b4f6d-4b01-42c6-84f3-3b434c46b2bd,Spreekbeurt - De voorzitter,...,Khadija,Arib,PvdA,lid Tweede Kamer,2020-10-13T14:01:13,2020-10-13T14:01:27,true,true,De voorzitter : Ik open de vergadering van de ...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
1,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,75a12a82-abe6-4fc2-a471-92e1ea0b65de,Spreekbeurt - De voorzitter,...,Khadija,Arib,PvdA,lid Tweede Kamer,2020-10-13T14:01:28,2020-10-13T14:01:48,true,false,De voorzitter : We beginnen zoals gebruikelijk...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
2,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,4703af9a-47ab-4e47-9bc2-a601d0da7352,Spreekbeurt - Lodders,...,Helma,Lodders,VVD,lid Tweede Kamer,2020-10-13T14:01:48,2020-10-13T14:03:41,false,false,Mevrouw Lodders (VVD): Voorzitter. In Nederlan...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
3,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,eb1cf359-d659-4877-b23d-8c23152a1a44,Spreekbeurt - Staatssecretaris Vijlbrief,...,Hans,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:03:41,2020-10-13T14:07:49,false,false,"Staatssecretaris Vijlbrief : Voorzitter, dank ...",verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...
4,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,Plenair,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vragenuur,eb1cf359-d659-4877-b23d-8c23152a1a44,Spreekbeurt - Staatssecretaris Vijlbrief,...,Hans,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:07:49,2020-10-13T14:08:58,false,false,Staatssecretaris Vijlbrief : Voorzitter. Laat ...,verslagen_xml\94fbdc2b-fd65-4610-a340-000018d7...


In [35]:
speeches_df.to_csv("speeches.csv", index=False)

In [36]:
speeches_df["is_voorzitter"] = speeches_df["is_voorzitter"] == "true"
speeches_no_chair = speeches_df[~speeches_df["is_voorzitter"]].copy()

# select useful columns
cols = [
    "vergadering_id",
    "vergadering_titel",
    "vergadering_datum",
    "activiteit_id",
    "activiteit_titel",
    "activiteit_onderwerp",
    "speaker_name",
    "speaker_party",
    "speaker_role",
    "speech_begin",
    "speech_end",
    "speech_text",
]

speeches_clean = speeches_no_chair[cols].copy()
speeches_clean.head()

,vergadering_id,vergadering_titel,vergadering_datum,activiteit_id,activiteit_titel,activiteit_onderwerp,speaker_name,speaker_party,speaker_role,speech_begin,speech_end,speech_text
2,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Lodders,VVD,lid Tweede Kamer,2020-10-13T14:01:48,2020-10-13T14:03:41,Mevrouw Lodders (VVD): Voorzitter. In Nederlan...
3,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:03:41,2020-10-13T14:07:49,"Staatssecretaris Vijlbrief : Voorzitter, dank ..."
4,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:07:49,2020-10-13T14:08:58,Staatssecretaris Vijlbrief : Voorzitter. Laat ...
5,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:08:58,2020-10-13T14:10:31,Staatssecretaris Vijlbrief : Voorzitter. De De...
6,71ab8f79-1c3b-4070-80cb-66c0e86ee1da,"13e vergadering, dinsdag 13 oktober 2020",2020-10-13T00:00:00,4f0f8f31-64b9-4709-a777-a8e70eb36886,Vragenuur,Vragenuur,Vijlbrief,,staatssecretaris van Financiën - Fiscaliteit e...,2020-10-13T14:10:31,2020-10-13T14:11:26,Staatssecretaris Vijlbrief : Als mevrouw het g...


In [37]:
speeches_clean.to_csv("speeches_clean.csv", index=False)

In [38]:
speeches_clean["vergadering_datum"] = pd.to_datetime(speeches_clean["vergadering_datum"], errors="coerce")
speeches_clean["speech_begin"] = pd.to_datetime(speeches_clean["speech_begin"], errors="coerce")
speeches_clean["speech_end"] = pd.to_datetime(speeches_clean["speech_end"], errors="coerce")

In [39]:
speeches_clean["speech_duration_seconds"] = (
    speeches_clean["speech_end"] - speeches_clean["speech_begin"]
).dt.total_seconds()

In [40]:
speeches_df[["speaker_name", "speaker_party", "speech_text"]].head(10)

,speaker_name,speaker_party,speech_text
0,Arib,PvdA,De voorzitter : Ik open de vergadering van de ...
1,Arib,PvdA,De voorzitter : We beginnen zoals gebruikelijk...
2,Lodders,VVD,Mevrouw Lodders (VVD): Voorzitter. In Nederlan...
3,Vijlbrief,,"Staatssecretaris Vijlbrief : Voorzitter, dank ..."
4,Vijlbrief,,Staatssecretaris Vijlbrief : Voorzitter. Laat ...
5,Vijlbrief,,Staatssecretaris Vijlbrief : Voorzitter. De De...
6,Vijlbrief,,Staatssecretaris Vijlbrief : Als mevrouw het g...
7,Vijlbrief,,Staatssecretaris Vijlbrief : Dat het niet gebe...
8,Vijlbrief,,Staatssecretaris Vijlbrief : Ik herhaal het to...
9,Vijlbrief,,Staatssecretaris Vijlbrief : Daar kan ik heel ...


In [41]:
speeches_df.to_excel("speeches.xlsx", index=False)